# Generate Monolayer Summary CSVs

This notebook reads PhysiCell monolayer `.mat` output files, extracts cell positions for each sampled timepoint, computes monolayer summary metrics such as cell count and colony diameter, and saves the aggregated results to CSV files for downstream analysis.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.spatial import ConvexHull, distance_matrix
from scipy.io import loadmat
from pathlib import Path


In [ ]:
base_path = "/home/tntiniak/Work/observatory_benchmark/PhysiCell/"
output_folder = base_path + "results/output_monolayer/"
files = sorted(Path(output_folder).glob('*output*_cells.mat'))

In [ ]:
# Loop through files
interval = 60
i=0
df_cell = pd.DataFrame()
results = []

for file in files:
    mat = loadmat(file)
    cell_data = mat['cells'][[0, 1, 2, 3]]
    df_mat = pd.DataFrame(cell_data.T, columns=['id', 'x', 'y', 'z'])
    df_mat['dt'] =i * interval
    df_cell = pd.concat([df_cell, df_mat], ignore_index=True)
    x = df_mat['x'].values
    y = df_mat['y'].values
    cell_count = len(x)
    if cell_count < 2:
        diameter = 0.0
    else:
        try:
            points = np.column_stack((x, y))
            hull = ConvexHull(points)
            hull_points = points[hull.vertices]
            distances = distance_matrix(hull_points, hull_points)
            diameter = distances.max()
        except:
            diameter = 0.0
    
    i += 1
    results.append({
    'timestep': i * interval,
    'cell_count': cell_count,
    'diameter': diameter
})
results

In [ ]:
df_summary = pd.DataFrame(results)
df_summary.to_csv(output_folder + "monolayer_t_diam.csv", index=False)